# Lab 4B: Production drills
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 4 · about 20 minutes**

**Objectives**
- Compare a pinned model with a candidate before switching
- Fall back cleanly when the primary model is overloaded
- Make retried side effects safe with idempotency keys
- Write team configuration for Claude Code

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

## Drill 1: a model canary
A tiny labelled set, run on the pinned model and a candidate. Day 11 builds proper evaluations.

In [ ]:
LABELLED = [("Whole town has no mobile data since the storm.", "outage"), ("Please send me last month's invoice again.", "billing"),
            ("My router keeps rebooting every hour.", "device"), ("I forgot my account password.", "account"),
            ("Charged a roaming fee but I never left the city.", "billing"), ("TV box shows error E-42 on start.", "device"),
            ("Street-wide fiber cut near the metro station.", "outage"), ("How do I add a second user to my account?", "account")]
SCHEMA = {"type": "object", "properties": {"category": {"type": "string", "enum": ["billing", "outage", "device", "account", "other"]}},
          "required": ["category"], "additionalProperties": False}
def run_eval(model):
    hits, t0, out_tokens = 0, time.perf_counter(), 0
    for text, label in LABELLED:
        r = client.messages.create(model=model, max_tokens=100, system="Classify the telecom ticket.",
                                   output_config={"format": {"type": "json_schema", "schema": SCHEMA}},
                                   messages=[{"role": "user", "content": text}])
        hits += json.loads(text_of(r))["category"] == label; out_tokens += r.usage.output_tokens
    return hits / len(LABELLED), (time.perf_counter() - t0) / len(LABELLED), out_tokens
for name, model in [("pinned", config.MODEL), ("candidate", config.FAST_MODEL)]:
    acc, lat, toks = run_eval(model)
    print(f"{name:9s} {model:30s} accuracy={acc:.0%} avg_latency={lat:.2f}s output_tokens={toks}")
print("\nDecision rule: switch only if accuracy holds on YOUR labelled set and cost or latency improves. Eight examples is too few to decide.")

## Drill 2: fall back on overload
Real 529 errors cannot be triggered on demand, so a simulator raises them.

In [ ]:
class SimulatedStatusError(Exception):
    def __init__(self, status_code): super().__init__(f"simulated HTTP {status_code}"); self.status_code = status_code

def primary_call(messages, overloaded):
    if overloaded:
        raise SimulatedStatusError(529)
    return client.messages.create(model=config.MODEL, max_tokens=200, messages=messages)

def answer_with_fallback(messages, overloaded=False):
    try:
        return primary_call(messages, overloaded), "primary"
    except SimulatedStatusError as e:
        if e.status_code not in (529, 500):
            raise
        r = client.messages.create(model=config.FAST_MODEL, max_tokens=200, messages=messages)
        return r, "fallback"
q = [{"role": "user", "content": "In one sentence, why might a router reboot by itself?"}]
for overloaded in (False, True):
    r, path = answer_with_fallback(q, overloaded)
    print(f"overloaded={overloaded!s:5s} path={path:8s} model={r.model} | {text_of(r)[:100]}")

Log which path served each request. A fallback should be visible in metrics, not silent.

## Drill 3: idempotent side effects

In [ ]:
import uuid
LEDGER, SEEN = [], {}
def issue_credit(account_id, amount, idempotency_key):
    if idempotency_key in SEEN:                # repeat of an action already done
        return SEEN[idempotency_key]
    entry = {"credit_id": f"CR-{len(LEDGER) + 1:04d}", "account": account_id, "amount": amount}
    LEDGER.append(entry); SEEN[idempotency_key] = entry
    return entry

def flaky_network(fn, *args):
    """Simulate a timeout AFTER the credit service processed the request, then a retry."""
    fn(*args)                                   # first attempt succeeds server-side, but the response is lost
    return fn(*args)                            # the client retries

key = str(uuid.uuid4())                         # generated ONCE per logical action
flaky_network(issue_credit, "C-1002", 74.99, key)
print("credits with a reused key:", LEDGER)
LEDGER.clear(); SEEN.clear()
flaky_network(lambda a, m, _: issue_credit(a, m, str(uuid.uuid4())), "C-1002", 74.99, None)
print("credits with a new key per attempt:", LEDGER)

## Drill 4: Claude Code configuration for a team repo
Shared settings are committed; personal settings are not.

In [ ]:
import pathlib
repo = pathlib.Path("team_repo"); (repo / ".claude").mkdir(parents=True, exist_ok=True)
(repo / "CLAUDE.md").write_text("""# Billing service
- Run tests with: python -m pytest -q
- Money values use Decimal, never float
- Model IDs live only in appcfg.py
""")
(repo / ".claude" / "settings.json").write_text(json.dumps({
    "permissions": {"allow": ["Bash(python -m pytest:*)"], "deny": ["Read(./.env)", "Read(./secrets/**)"]}}, indent=2))
(repo / ".claude" / "settings.local.json").write_text(json.dumps({"permissions": {"allow": ["Bash(make docs)"]}}, indent=2))
(repo / ".gitignore").write_text(".env\n.claude/settings.local.json\n")
for p in sorted(repo.rglob("*")):
    if p.is_file():
        print(f"{str(p):38s} committed={'no' if p.name in ('settings.local.json', '.env') else 'yes'}")
print("\nsettings.json is valid JSON:", bool(json.loads((repo / '.claude' / 'settings.json').read_text())))

## You should now have
- [ ] A pinned-versus-candidate comparison
- [ ] A visible fallback path on overload
- [ ] Idempotent credits under retry
- [ ] Shared and personal Claude Code settings

## Check yourself
1. What must be true before switching models?
2. Why generate the idempotency key once per action?
3. Which Claude Code settings file is gitignored?